In [2]:
%pip install langchain langchain-community langchain-openai langchain-text-splitters langchain-chroma fastembed pypdf python-dotenv

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.1-py3-none-any.whl.metadata (2.6 kB)
Using cached langchain_chroma-1.1.0-py3-none-any.whl (12 kB)
Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl (23.5 MB)
Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.1-py3-none-any.whl (19 kB)

   ----- ---------------------------------- 1/7 [py-rust-stemmers]
   ----------- ---------------------------- 2/7 [loguru]
   ----------- ---------------------------- 2/7 [loguru]
   ----------- ---------------------------- 2/7 [loguru]
   ------------- ----------------- 3/7 [opentelemetry-exporter-otlp-proto-grpc]
   ---------------------- ----------------- 4/7 [fastembed]
   ---------------------- ----------------- 4/7 [fastembed]
   ---------------------- ----------------- 4/7 [fastembed]
   ---------------------- ----------------- 4/7 [fast

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
# Step 1: Load the PDF
from langchain_community.document_loaders import PyPDFLoader

docs = PyPDFLoader("data/hotel_info.pdf").load()
print("Number of pages:", len(docs))
print(docs[0].page_content[:800])

C:\Users\HS TRADER\AppData\Local\Temp\ipykernel_29432\2936453732.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 3
Azure Bay Grand Hotel – Guest Information Guide 
About the Hotel 
Azure Bay Grand Hotel is a five-star luxury beachfront hotel at 88 Coral Bay Avenue, Marina District, 
Seaview City. It opened in 2012 and was fully renovated in 2023. The hotel has 320 rooms and 
suites across 24 floors, a 400-metre private beach, and views over Azure Bay. It is 25 minutes by car 
from Seaview International Airport and 10 minutes from the city centre and the Old Town market. 
The reception and concierge desks are open 24 hours a day. Staff speak English, Arabic, French and 
Urdu. 
Contact Details 
For reservations, guests can call +1 555 0142 or email reservations@azurebaygrand.example. The 
front desk is at +1 555 0100. Group and event bookings go to events@azurebaygrand.example. The 
website is www.azureb


In [2]:
# Step 2: Split into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(docs)
print("Number of chunks:", len(chunks))

Number of chunks: 14


In [3]:
# Step 3: Embeddings with a local model (FastEmbed, no PyTorch, no Azure)
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"                # use normal download (xet hangs on Windows)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # hide the symlinks warning

from langchain_community.embeddings import FastEmbedEmbeddings

embeddings = FastEmbedEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    cache_dir="models",   # save the model in the project folder, not in Temp
)

vector = embeddings.embed_query("Can I cancel my booking?")
print("Embedding length:", len(vector))   # should be 384
print("First 5 numbers:", vector[:5])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding length: 384
First 5 numbers: [-0.011430677026510239, -0.04928068816661835, 0.0426161102950573, 0.008529860526323318, -0.05108622834086418]


In [4]:
# Step 4: Store chunks in ChromaDB
import shutil
from langchain_chroma import Chroma

shutil.rmtree("chroma_db", ignore_errors=True)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="hotel_info",
    persist_directory="chroma_db",
)
print("Stored chunks:", vectorstore._collection.count())   # 14

Stored chunks: 14


In [5]:
# Step 5: Test retrieval
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("Is it free to call off my reservation?")
for i, doc in enumerate(results):
    print(f"===== Result {i+1} | page {doc.metadata['page']} =====")
    print(doc.page_content[:400], "\n")

===== Result 1 | page 0 =====
card is needed for a refundable security deposit of PKR 28,000 per night, up to PKR 140,000. 
Reservation Policy 
Reservations can be made on the hotel website, by phone, by email or at the front desk. A booking 
is confirmed only after the guest receives a confirmation email with a booking number. Standard 
rate bookings need no prepayment, and payment is taken at check-in. Non-refundable saver r 

===== Result 2 | page 1 =====
Cancellation and No-Show Policy 
For standard rates, cancellation is free up to 48 hours before the check-in date. If a guest cancels 
within 48 hours of arrival, they are charged one night. A guest who does not arrive is charged one 
night and the rest of the booking is cancelled. Non-refundable saver rates cannot be cancelled, 
changed or refunded. Group bookings of 5 or more rooms must be cance 

===== Result 3 | page 2 =====
Visitors must leave guest rooms by 10:00 PM. Guests must wear proper swimwear in the pools, and 
childre

In [6]:
# Step 6: RAG chain (Chroma + Azure gpt-4.1-mini)
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("CHAT_DEPLOYMENT", "gpt-4.1-mini"),
    base_url=os.environ["AZURE_OPENAI_BASE_URL"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the reservation assistant for Azure Bay Grand Hotel. "
     "Answer ONLY using the hotel information below. If the answer is not there, "
     "say you don't have that information and suggest contacting reservations.\n\n"
     "Hotel information:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print(rag_chain.invoke("How much is a family room and can I cancel for free?"))
print("\n---\n")
print(rag_chain.invoke("Do you have a golf course?"))

I don't have information about the rates for a family room at Azure Bay Grand Hotel. Regarding cancellation, for standard rates, cancellation is free up to 48 hours before the check-in date. If you cancel within 48 hours of arrival, you will be charged for one night. Non-refundable saver rates cannot be cancelled, changed, or refunded. For more details about room rates, please contact reservations.

---

The hotel information does not mention a golf course. For more details, please contact reservations.
